# B4. Data scaling

Does more data still help?

See `docs/plans/experiments-plan.md`, entry B4. Results: `artifacts/results/erosion_uniform/`.

## Run

From the repo root. `uv run erosion-sim list` shows what is already trained and evaluated.

**1. Data.** Skip any that already exist.

Local:

```sh
(cd datagen && uv run erosion-datagen generate erosion_master/main_256)
```

Cluster:

```sh
sbatch slurm/datagen.sbatch erosion_master/main_256
```

**2. Train**, in this order. Each resumes if stopped; a finished model is skipped.

Local:

```sh
uv run erosion-sim train erosion_runs_333
uv run erosion-sim train erosion_runs_667
uv run erosion-sim train erosion_runs_1333
uv run erosion-sim train erosion_base
```

Cluster:

```sh
sbatch slurm/train.sbatch erosion_runs_333
sbatch slurm/train.sbatch erosion_runs_667
sbatch slurm/train.sbatch erosion_runs_1333
sbatch slurm/train.sbatch erosion_base
```

**3. Evaluate**, once training has finished. It refuses to replace existing results.

Local:

```sh
uv run erosion-sim evaluate erosion_uniform erosion_runs_333
uv run erosion-sim evaluate erosion_uniform erosion_runs_667
uv run erosion-sim evaluate erosion_uniform erosion_runs_1333
uv run erosion-sim evaluate erosion_uniform erosion_base
uv run erosion-sim evaluate erosion_uniform no_change
uv run erosion-sim evaluate erosion_uniform analytical
```

Cluster:

```sh
sbatch slurm/evaluate.sbatch erosion_uniform erosion_runs_333
sbatch slurm/evaluate.sbatch erosion_uniform erosion_runs_667
sbatch slurm/evaluate.sbatch erosion_uniform erosion_runs_1333
sbatch slurm/evaluate.sbatch erosion_uniform erosion_base
sbatch slurm/evaluate.sbatch erosion_uniform no_change
sbatch slurm/evaluate.sbatch erosion_uniform analytical
```

Then run the cells below.

How to read the tables and plots: [`docs/guides/erosion.md`](../../docs/guides/erosion.md).

In [ ]:
import json

from erosion_sim.erosion import plots
from erosion_sim.evaluation.results import load
from erosion_sim.paths import model_dir
from erosion_sim.training.train import plot_history

EVALUATION = "erosion_uniform"  # configs/evaluations/erosion_uniform.yaml
MODELS = ["erosion_runs_333", "erosion_runs_667", "erosion_runs_1333", "erosion_base"]  # trained models
BASELINES = ["no_change", "analytical"]  # configs/predictors/
PREDICTORS = MODELS + BASELINES

results = load(EVALUATION, *PREDICTORS)
{p: {k: m[k] for k in ("time", "code", "slurm_job")} for p, m in results.manifest["predictors"].items()}

## Training

In [ ]:
for name in MODELS:
    print(name, model_dir(name))
    plot_history(json.loads((model_dir(name) / "history.json").read_text()))

## Every metric, over all test pairs

In [ ]:
plots.table(results)

## Breakdowns

Each factor with more than one group among the test pairs.

In [ ]:
for factor in results.scalars["by"]:
    plots.by_table(results, factor, "skill")
    print()

In [ ]:
for factor in results.scalars["by"]:
    plots.plot_by(results, factor, "skill")

## Detail and examples

In [ ]:
plots.plot_spectrum(results)
plots.plot_examples(results)

## Conclusion

_Written after reading the results above._